# Profile tables for a synthetic Genie demo

This notebook reads the tables you list and writes **statistics only**: column types, share of empty values,
typical ranges, most common categories, and how tables link to each other. It **never exports rows** or
personal values (names, emails, phones, addresses, IDs...). It only reads your tables: it creates, changes
or deletes nothing outside the output folder you choose.

**How to use it**
1. Attach the notebook to a cluster (Databricks Runtime 15.4 LTS or newer, or serverless).
2. Edit **cell 2 (Configuration)**: the tables and an output folder you can write to.
3. Click **Run all**. It takes a few minutes per table.
4. Read the files in `<output folder>/review/` (see the review notes further down).
5. Run the **last cell** to zip the reviewed files, download the zip and send only that file.

You need read access to the tables, write access to the output folder, and the cluster must be able to
install `pydantic` from PyPI (cell 1).

## 1. Install the one dependency (pydantic)

In [ ]:
%pip install -q "pydantic>=2.7"

In [ ]:
dbutils.library.restartPython()  # loads the freshly installed package

In [ ]:
# ============================== 2. CONFIGURATION: edit, then run =================================

# Tables to profile: full names, or catalog.schema.* for every table of a schema.
TABLES = [
    "main.sales.*",
]

# Folder where the results are written. A Unity Catalog volume you can write to, e.g.
#   "/Volumes/<catalog>/<schema>/<volume>/synthgen"
# or your own workspace folder, e.g. "/Workspace/Users/<your.email>/synthgen_output"
OUTPUT_FOLDER = "/Volumes/main/sandbox/files/synthgen"

# Optional: columns whose values must never leave, as catalog.schema.table.column (* allowed).
# Personal data (names, emails, phones, addresses, IDs, IBANs, birth dates, IPs...) and columns with
# a Unity Catalog PII tag are already detected automatically. Add anything else confidential here.
EXTRA_SENSITIVE_COLUMNS = [
    # "main.sales.customers.internal_notes",
]

# Optional: advanced settings. Leave empty for the defaults. Examples:
#   {"privacy": {"k": 20}}                          hide any value shared by fewer than 20 rows (default 10)
#   {"profiling": {"sample_fraction": 0.1}}         compute statistics on a 10% sample (very large tables)
ADVANCED_SETTINGS = {}

In [ ]:
# ============================== 3. ENGINE: run, do not edit ======================================
# The profiling code (synthgen package, embedded so nothing has to be uploaded). It is plain Python:
# feel free to read it. It only defines functions; nothing is read or written until cell 4.

import sys
import types


def _load(name, source, package=False):
    module = types.ModuleType(name)
    module.__file__ = "<synthgen notebook: " + name + ">"
    if package:
        module.__path__ = []
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)


_load("synthgen", r'''
"""synthgen: profile tables into privacy-filtered statistics, regenerate synthetic look-alikes."""

__version__ = "0.1.0"
''', package=True)

_load("synthgen.config", r'''
"""Dataset configuration.

Every threshold the toolkit uses lives here. Defaults are declared once on these models and
`config/example_dataset.yaml` spells all of them out, so a dataset owner can see and tune each one.
Code never hard-codes a threshold; it reads it from a `SynthgenConfig` instance.
"""

from __future__ import annotations

from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# Hard floor for k. The plan allows configuring k but never below 5.
MIN_K = 5


class _Section(BaseModel):
    model_config = ConfigDict(extra="forbid")


class SourceConfig(_Section):
    tables: list[str] = Field(default_factory=list, description="Explicit catalog.schema.table names.")
    table_patterns: list[str] = Field(
        default_factory=list, description="Glob patterns on catalog.schema.table, e.g. 'main.sales.*'."
    )
    exclude_patterns: list[str] = Field(default_factory=list)
    include_views: bool = False

    @model_validator(mode="after")
    def _non_empty(self) -> SourceConfig:
        if not self.tables and not self.table_patterns:
            raise ValueError("source needs at least one of 'tables' or 'table_patterns'")
        for name in self.tables:
            if name.count(".") != 2:
                raise ValueError(f"table '{name}' must be fully qualified as catalog.schema.table")
        return self


class OutputConfig(_Section):
    root: str = Field(description="Folder the profiler may write to. Profiles land in <root>/review/.")


DEFAULT_PII_NAME_PATTERNS: dict[str, list[str]] = {
    # semantic type -> regexes matched against the lower-cased column name.
    "person_name": [
        r"(^|_)(first|last|full|given|family|middle|maiden|customer|contact|person|user|owner|holder)_?name($|_)",
        r"^name$",
        r"(^|_)(surname|forename|firstname|lastname)($|_)",
    ],
    "email": [r"e_?mail"],
    "phone": [r"phone", r"mobile", r"(^|_)(tel|telephone|fax|msisdn)($|_)"],
    # ip_address precedes address so "ip_address" gets the more specific semantic type.
    "ip_address": [r"(^|_)ip($|_)", r"ip_?addr"],
    "address": [r"address", r"(^|_)(street|addr|address_line)($|_)"],
    "postcode": [r"postcode", r"postal", r"(^|_)zip($|_|code)"],
    "iban": [r"iban", r"(^|_)(swift|bic)($|_)", r"(account|card)_?(number|no|num|nr)", r"(^|_)ccn($|_)"],
    "national_id": [
        r"(^|_)(ssn|dni|nif|nie|nir|cpf|nino|passport)($|_)",
        r"social_?security",
        r"(tax|national|personal)_?id",
    ],
    "birth_date": [r"birth", r"(^|_)dob($|_)"],
    "geo": [r"(^|_)(lat|lon|lng|latitude|longitude|geo|geolocation)($|_)"],
    "secret": [r"password", r"(^|_)(secret|token|api_?key)($|_)"],
    "special_category": [r"(^|_)(ethnic|ethnicity|religion|health|diagnosis|medical)($|_)"],
}

DEFAULT_ID_NAME_PATTERNS = [r"^id$", r"_id$", r"^id_", r"_key$", r"_uuid$", r"^uuid$", r"_no$", r"_nr$", r"^sku$"]


class PrivacyConfig(_Section):
    k: int = Field(10, description=f"Minimum group size. Never below {MIN_K}.")
    top_k_values: int = Field(50, ge=1, description="Max category labels exported per column.")
    clip_lower: float = Field(0.01, description="Quantile exported as 'min'. Must be >= 0.01.")
    clip_upper: float = Field(0.99, description="Quantile exported as 'max'. Must be <= 0.99.")
    min_count_for_distribution: int = Field(
        50, description="Non-null rows needed before quantiles, mean, std or length stats are exported."
    )
    pii_name_patterns: dict[str, list[str]] = Field(default_factory=lambda: dict(DEFAULT_PII_NAME_PATTERNS))
    pii_tag_patterns: list[str] = Field(
        default_factory=lambda: ["pii", "pii.*", "pii_*", "class.*", "sensitive", "personal*", "gdpr*"],
        description="Glob patterns matched against a UC tag name and against 'name=value'.",
    )
    sensitive_columns: list[str] = Field(
        default_factory=list, description="Glob patterns on catalog.schema.table.column forced to sensitive."
    )
    value_pii_min_share: float = Field(
        0.3, gt=0, le=1, description="Share of values matching an email/phone/IP/IBAN regex that marks a column PII."
    )
    allowed_labels: list[str] = Field(
        default_factory=list,
        description="Columns (glob on catalog.schema.table.column) whose category labels may leave. Used by the guard.",
    )

    @field_validator("k")
    @classmethod
    def _k_floor(cls, v: int) -> int:
        if v < MIN_K:
            raise ValueError(f"k must be >= {MIN_K}, got {v}")
        return v

    @model_validator(mode="after")
    def _bounds(self) -> PrivacyConfig:
        if self.clip_lower < 0.01 or self.clip_upper > 0.99 or self.clip_lower >= self.clip_upper:
            raise ValueError("clip_lower must be >= 0.01, clip_upper <= 0.99, and clip_lower < clip_upper")
        if self.min_count_for_distribution < self.k:
            raise ValueError("min_count_for_distribution must be >= k")
        return self


class ProfilingConfig(_Section):
    max_categorical_distinct: int = Field(200, ge=1)
    max_categorical_ratio: float = Field(0.5, gt=0, le=1)
    high_cardinality_ratio: float = Field(
        0.5, gt=0, le=1, description="String columns above this distinct ratio export no values."
    )
    quantiles: list[float] = Field(default_factory=lambda: [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    percentile_accuracy: int = Field(10000, ge=100)
    approx_distinct_rsd: float = Field(0.02, gt=0, le=0.1)
    pattern_top_n: int = Field(5, ge=1)
    pattern_max_length: int = Field(40, ge=1, description="Values longer than this get no format pattern.")
    max_year_months: int = Field(600, ge=1, description="Max year-month buckets exported per date column.")
    max_decimals: int = Field(6, ge=0)
    free_text_min_mean_length: float = Field(30.0, gt=0)
    id_name_patterns: list[str] = Field(default_factory=lambda: list(DEFAULT_ID_NAME_PATTERNS))
    sample_fraction: float = Field(1.0, gt=0, le=1)
    sample_fractions: dict[str, float] = Field(
        default_factory=dict, description="Per-table override, keyed by catalog.schema.table."
    )
    sample_seed: int = 42

    @field_validator("quantiles")
    @classmethod
    def _quantiles(cls, v: list[float]) -> list[float]:
        if any(q <= 0 or q >= 1 for q in v):
            raise ValueError("quantiles must be strictly between 0 and 1")
        return sorted(set(v))

    @field_validator("sample_fractions")
    @classmethod
    def _fractions(cls, v: dict[str, float]) -> dict[str, float]:
        for name, f in v.items():
            if not 0 < f <= 1:
                raise ValueError(f"sample fraction for {name} must be in (0, 1]")
        return v


class ForcedForeignKey(_Section):
    child: str = Field(description="catalog.schema.table.column")
    parent: str = Field(description="catalog.schema.table.column")


class RelationshipsConfig(_Section):
    fk_min_containment: float = Field(0.9, gt=0, le=1)
    name_similarity_threshold: float = Field(0.8, gt=0, le=1)
    pk_prefilter_ratio: float = Field(
        0.9, gt=0, le=1, description="Approx distinct ratio a column needs before an exact uniqueness check."
    )
    max_pk_pair_columns: int = Field(6, ge=2)
    children_quantiles: list[float] = Field(default_factory=lambda: [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    forced_pks: dict[str, list[str]] = Field(default_factory=dict, description="table -> key columns")
    forced_fks: list[ForcedForeignKey] = Field(default_factory=list)


class RulesConfig(_Section):
    date_order_min_rate: float = Field(0.9, gt=0.5, le=1)
    identity_min_rate: float = Field(0.95, gt=0.5, le=1)
    identity_sample_rows: int = Field(20000, ge=100)
    identity_abs_tolerance: float = Field(0.01, ge=0)
    identity_rel_tolerance: float = Field(1e-6, ge=0)
    max_identity_columns: int = Field(10, ge=3)


class SynthgenConfig(_Section):
    dataset: str
    source: SourceConfig
    output: OutputConfig
    metadata_provider: Literal["auto", "unity_catalog", "spark"] = "auto"
    privacy: PrivacyConfig = Field(default_factory=PrivacyConfig)
    profiling: ProfilingConfig = Field(default_factory=ProfilingConfig)
    relationships: RelationshipsConfig = Field(default_factory=RelationshipsConfig)
    rules: RulesConfig = Field(default_factory=RulesConfig)

    @classmethod
    def load(cls, path: str | Path) -> SynthgenConfig:
        import yaml  # only needed for YAML configs; notebooks configure in Python

        with open(path, encoding="utf-8") as fh:
            raw = yaml.safe_load(fh) or {}
        return cls.model_validate(raw)

    def sample_fraction_for(self, table: str) -> float:
        return self.profiling.sample_fractions.get(table, self.profiling.sample_fraction)
''')

_load("synthgen.profile_schema", r'''
"""Versioned schema for profile files: the only contract between the two workspaces.

Conventions used throughout:
- Shares of column values (categories, patterns, months, ...) are over the column's non-null values.
  `null_rate` and boolean shares are over all rows.
- A rate or share is `null` and its field name is listed in `suppressed` when the underlying count
  was between 1 and k-1. A share of exactly 0.0 means the count was 0.
- `min_clipped` / `max_clipped` are the configured clip quantiles (default 1st and 99th), never true extremes.
  Numeric mean and std are computed on values capped at those bounds, so outliers cannot inflate them.
- Quantile keys are the quantile written as a string, e.g. "0.5".

Run `python -m synthgen.profile_schema schemas/` to regenerate the JSON Schema files for reviewers.
"""

from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal, Union

from pydantic import BaseModel, ConfigDict, Field

SCHEMA_VERSION = "1.0"

Share = Annotated[float, Field(ge=0.0, le=1.0)]

Role = Literal["key", "foreign_key", "categorical", "numeric", "date", "text", "boolean", "other"]

SemanticType = Literal[
    "email",
    "phone",
    "person_name",
    "postcode",
    "address",
    "uuid",
    "ip_address",
    "iban",
    "national_id",
    "birth_date",
    "geo",
    "secret",
    "special_category",
    "identifier",
    "free_text",
    "short_text",
    "unknown",
]


class _Model(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True, allow_inf_nan=False)


class LengthStats(_Model):
    """String length statistics. Min and max are clipped quantiles of the length."""

    min_clipped: float | None = None
    mean: float | None = None
    max_clipped: float | None = None


class ValueShare(_Model):
    value: str
    share: Share


class PatternShare(_Model):
    pattern: str = Field(description="Letters mapped to A, digits to 9, punctuation and spaces kept.")
    share: Share


class NumericStats(_Model):
    min_clipped: float | None = None
    max_clipped: float | None = None
    mean: float | None = Field(None, description="Mean of values capped at min_clipped and max_clipped.")
    std: float | None = Field(None, description="Sample std of values capped at min_clipped and max_clipped.")
    quantiles: dict[str, float] = Field(default_factory=dict)
    zero_share: Share | None = None
    negative_share: Share | None = None
    is_integer: bool
    decimals: int = Field(ge=0)


class CategoricalStats(_Model):
    top_values: list[ValueShare] = Field(default_factory=list)
    other_share: Share = Field(description="Share of non-null values in labels not listed (rare or beyond top K).")
    labels_not_exported: int = Field(ge=0, description="Approximate number of labels folded into 'other'.")
    length: LengthStats = Field(default_factory=LengthStats)


class TextStats(_Model):
    """Exported for sensitive, identifier and high-cardinality columns. Never contains values."""

    length: LengthStats = Field(default_factory=LengthStats)
    patterns: list[PatternShare] = Field(default_factory=list)
    other_pattern_share: Share
    semantic_type: SemanticType


class DateStats(_Model):
    """Date and timestamp statistics. Dates are ISO strings; UTC timestamps end in 'Z'."""

    min_clipped: str | None = None
    max_clipped: str | None = None
    quantiles: dict[str, str] = Field(default_factory=dict)
    year_month_shares: dict[str, Share] = Field(default_factory=dict)
    other_year_month_share: Share = 0.0
    day_of_week_shares: dict[str, Share] = Field(default_factory=dict)
    # Timestamp only.
    hour_shares: dict[str, Share] | None = None
    always_midnight: bool | None = None
    timezone: str | None = None


class BooleanStats(_Model):
    true_share: Share | None = None
    false_share: Share | None = None


class ColumnProfile(_Model):
    name: str
    position: int = Field(ge=0)
    type: str = Field(description="Spark SQL type string.")
    nullable: bool
    comment: str | None = None
    tags: dict[str, str] = Field(default_factory=dict)
    is_partition: bool = False
    role: Role
    sensitive: bool
    sensitive_reasons: list[str] = Field(default_factory=list)
    null_rate: Share | None = None
    distinct_approx: int = Field(ge=0)
    distinct_ratio: float | None = Field(None, ge=0.0, description="distinct_approx / non-null count.")
    suppressed: list[str] = Field(default_factory=list)
    numeric: NumericStats | None = None
    categorical: CategoricalStats | None = None
    text: TextStats | None = None
    date: DateStats | None = None
    boolean: BooleanStats | None = None


class DateOrderRule(_Model):
    kind: Literal["date_order"] = "date_order"
    before: str
    after: str
    match_rate: Share | None = Field(description="Share of rows with both set where before <= after.")
    suppressed: list[str] = Field(default_factory=list)


class NumericIdentityRule(_Model):
    kind: Literal["numeric_identity"] = "numeric_identity"
    target: str
    operation: Literal["product", "sum"]
    operands: list[str] = Field(min_length=2, max_length=2)
    match_rate: Share | None
    sample_fraction: float = Field(gt=0, le=1)
    suppressed: list[str] = Field(default_factory=list)


Rule = Annotated[Union[DateOrderRule, NumericIdentityRule], Field(discriminator="kind")]


class DeclaredForeignKey(_Model):
    name: str | None = None
    columns: list[str] = Field(min_length=1)
    parent_table: str
    parent_columns: list[str] = Field(min_length=1)


class TableProfile(_Model):
    schema_version: Literal["1.0"] = SCHEMA_VERSION
    generated_at: datetime
    profiler_version: str
    source_table: str
    row_count: int = Field(ge=0)
    sample_fraction: float = Field(gt=0, le=1)
    k_threshold: int = Field(ge=5)
    comment: str | None = None
    tags: dict[str, str] = Field(default_factory=dict)
    primary_key: list[str] | None = Field(None, description="Declared primary key, if any.")
    foreign_keys: list[DeclaredForeignKey] = Field(default_factory=list, description="Declared foreign keys.")
    partition_columns: list[str] = Field(default_factory=list)
    columns: list[ColumnProfile]
    rules: list[Rule] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)


KeySource = Literal["declared", "config", "inferred"]


class PrimaryKey(_Model):
    table: str
    columns: list[str] = Field(min_length=1)
    source: KeySource
    alternatives: list[list[str]] = Field(default_factory=list, description="Other unique column sets found.")


class ChildrenPerParent(_Model):
    quantiles: dict[str, float] = Field(default_factory=dict)
    zero_children_share: Share | None = None


class ForeignKey(_Model):
    child_table: str
    child_columns: list[str] = Field(min_length=1)
    parent_table: str
    parent_columns: list[str] = Field(min_length=1)
    source: KeySource
    name_score: float | None = None
    containment: Share | None = Field(None, description="Share of non-null child rows whose value is in the parent key.")
    orphan_rate: Share | None = Field(None, description="Share of non-null child rows whose value is not in the parent key.")
    orphan_value_share: Share | None = Field(
        None, description="Share of distinct non-null child values that are not in the parent key."
    )
    children_per_parent: ChildrenPerParent | None = None
    suppressed: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)


class Relationships(_Model):
    schema_version: Literal["1.0"] = SCHEMA_VERSION
    generated_at: datetime
    profiler_version: str
    k_threshold: int = Field(ge=5)
    tables: list[str]
    primary_keys: list[PrimaryKey] = Field(default_factory=list)
    foreign_keys: list[ForeignKey] = Field(default_factory=list)


def write_json_schemas(out_dir: str | Path) -> list[Path]:
    import json

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = []
    for model, name in [(TableProfile, "table_profile"), (Relationships, "relationships")]:
        path = out / f"{name}.v{SCHEMA_VERSION}.schema.json"
        path.write_text(json.dumps(model.model_json_schema(), indent=2) + "\n", encoding="utf-8")
        written.append(path)
    return written


if __name__ == "__main__":
    for p in write_json_schemas(sys.argv[1] if len(sys.argv) > 1 else "schemas"):
        print(p)
''')

_load("synthgen.io", r'''
"""Reading and writing profile files.

All profile output goes through `ProfileStore`, which only writes inside its own root folder and
validates every file against the profile schema on write and on read.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import TypeVar

from pydantic import BaseModel

from synthgen.profile_schema import Relationships, TableProfile

REVIEW_DIR = "review"
RELATIONSHIPS_FILE = "relationships.json"
PROFILE_SUFFIX = ".profile.json"

M = TypeVar("M", bound=BaseModel)


class OutputPathError(RuntimeError):
    pass


def dump_json(model: BaseModel) -> str:
    """Human-readable, stable JSON: indented, field order fixed by the schema."""
    return json.dumps(model.model_dump(mode="json"), indent=2, ensure_ascii=False) + "\n"


class ProfileStore:
    """A folder of profile files, e.g. <output_root>/review."""

    def __init__(self, folder: str | Path):
        self.folder = Path(folder).resolve()

    @classmethod
    def review(cls, output_root: str | Path) -> ProfileStore:
        return cls(Path(output_root) / REVIEW_DIR)

    def _path(self, filename: str) -> Path:
        if "/" in filename or "\\" in filename or filename.startswith("."):
            raise OutputPathError(f"invalid profile file name: {filename!r}")
        path = (self.folder / filename).resolve()
        if path.parent != self.folder:
            raise OutputPathError(f"{path} is outside {self.folder}")
        return path

    def _write(self, filename: str, model: BaseModel) -> Path:
        text = dump_json(model)
        # Round-trip through the schema so nothing invalid is ever written.
        type(model).model_validate_json(text)
        self.folder.mkdir(parents=True, exist_ok=True)
        path = self._path(filename)
        path.write_text(text, encoding="utf-8")
        return path

    def write_table_profile(self, profile: TableProfile) -> Path:
        return self._write(profile.source_table + PROFILE_SUFFIX, profile)

    def write_relationships(self, rel: Relationships) -> Path:
        return self._write(RELATIONSHIPS_FILE, rel)

    def _read(self, filename: str, model: type[M]) -> M:
        return model.model_validate_json(self._path(filename).read_text(encoding="utf-8"))

    def read_table_profile(self, table: str) -> TableProfile:
        return self._read(table + PROFILE_SUFFIX, TableProfile)

    def read_relationships(self) -> Relationships:
        return self._read(RELATIONSHIPS_FILE, Relationships)

    def table_profiles(self) -> list[TableProfile]:
        return [
            TableProfile.model_validate_json(p.read_text(encoding="utf-8"))
            for p in sorted(self.folder.glob("*" + PROFILE_SUFFIX))
        ]
''')

_load("synthgen.notebook_helpers", r'''
"""Glue for the Databricks notebooks: turn the configuration cell into a config object."""

from __future__ import annotations

from typing import Any

from synthgen.config import SynthgenConfig


def split_list(value: str | list[str] | tuple[str, ...]) -> list[str]:
    items = value.replace("\n", ",").split(",") if isinstance(value, str) else list(value)
    return [item.strip() for item in items if item and item.strip()]


def _merge(base: dict, extra: dict) -> dict:
    for key, value in extra.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _merge(base[key], value)
        else:
            base[key] = value
    return base


def config_from_notebook(
    tables: str | list[str],
    output_folder: str,
    extra_sensitive_columns: str | list[str] = (),
    advanced_settings: dict[str, Any] | None = None,
) -> SynthgenConfig:
    """Defaults, plus the few values a person fills in at the top of the notebook.

    tables: catalog.schema.table names or patterns such as main.sales.*
    extra_sensitive_columns: catalog.schema.table.column names (globs allowed)
    advanced_settings: optional overrides with the same structure as config/example_dataset.yaml,
        e.g. {"privacy": {"k": 20}, "profiling": {"sample_fraction": 0.1}}
    """
    patterns = split_list(tables)
    if not patterns:
        raise ValueError('TABLES is empty: list at least one table, e.g. ["main.sales.*"]')
    for p in patterns:
        if p.count(".") != 2:
            raise ValueError(f"'{p}' in TABLES must be catalog.schema.table (use * for all tables of a schema)")
    if not output_folder or not output_folder.strip():
        raise ValueError('OUTPUT_FOLDER is empty, e.g. "/Volumes/main/sandbox/files/synthgen"')
    raw: dict[str, Any] = {
        "dataset": "genie_demo",
        "source": {"table_patterns": patterns},
        "output": {"root": output_folder.strip()},
        "privacy": {"sensitive_columns": split_list(extra_sensitive_columns)},
    }
    return SynthgenConfig.model_validate(_merge(raw, advanced_settings or {}))
''')

_load("synthgen.profiler", r'''
''', package=True)

_load("synthgen.profiler._safe", r'''
"""The only place profiler code may bring Spark results to the driver.

Shipped profiler code must not call collect/show/take/toPandas/display anywhere else; a test scans
the source to enforce it. Every DataFrame passed here must be an aggregate (or catalog metadata) that
was built so that no individual row value can survive: callers filter buckets below k inside Spark.
The row cap is a backstop against a caller accidentally passing a row-level DataFrame.
"""

from __future__ import annotations

from typing import Any

from pyspark.sql import DataFrame, Row


class RowLevelOutputError(RuntimeError):
    pass


def collect_aggregate(df: DataFrame, *, max_rows: int, what: str) -> list[Row]:
    rows = df.limit(max_rows + 1).collect()
    if len(rows) > max_rows:
        raise RowLevelOutputError(f"{what}: expected at most {max_rows} aggregate rows, got more")
    return rows


def collect_single(df: DataFrame, *, what: str) -> dict[str, Any]:
    rows = collect_aggregate(df, max_rows=1, what=what)
    if not rows:
        raise RowLevelOutputError(f"{what}: aggregate returned no row")
    return rows[0].asDict(recursive=True)


def collect_metadata(df: DataFrame, *, max_rows: int, what: str) -> list[Row]:
    """Catalog / information_schema queries only. Never pass a DataFrame built on a source table."""
    return collect_aggregate(df, max_rows=max_rows, what=what)
''')

_load("synthgen.profiler.detect", r'''
"""Pure-Python heuristics: PII detection, identifier names, roles, semantic types, FK name matching.

Nothing here touches Spark or data; it works on names, types, tags and aggregate statistics only.
"""

from __future__ import annotations

import fnmatch
import re
from dataclasses import dataclass, field
from difflib import SequenceMatcher

from synthgen.config import SynthgenConfig

# Value regexes evaluated in Spark (Java regex syntax); shares feed semantic typing and value-based PII.
VALUE_REGEXES: dict[str, str] = {
    "email": r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$",
    "uuid": r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$",
    # 8-15 digits with common separators; IPv4 addresses and ISO dates excluded.
    "phone": r"^(?!([0-9]{1,3}\.){3}[0-9]{1,3}$)(?![0-9]{4}-[0-9]{2}-[0-9]{2})(?=(\D*\d){8,15}\D*$)\+?[0-9(][0-9 ().-]{6,22}[0-9]$",
    "ip_address": r"^([0-9]{1,3}\.){3}[0-9]{1,3}$|^[0-9a-fA-F:]{2,39}:[0-9a-fA-F]{1,4}$",
    "iban": r"^[A-Z]{2}[0-9]{2}[A-Z0-9]{10,30}$",
}
# Value patterns strong enough on their own to mark a column sensitive.
VALUE_PII_TYPES = ("email", "phone", "ip_address", "iban")

INTEGER_TYPES = {"tinyint", "smallint", "int", "bigint", "byte", "short", "integer", "long"}
FLOAT_TYPES = {"float", "double", "real"}


def type_family(spark_type: str) -> str:
    """Coarse type family: integer, decimal, float, string, date, timestamp, boolean, other."""
    t = spark_type.lower()
    if t in INTEGER_TYPES:
        return "integer"
    if t.startswith("decimal"):
        m = re.match(r"decimal\((\d+),\s*(\d+)\)", t)
        return "integer" if m and m.group(2) == "0" else "decimal"
    if t in FLOAT_TYPES:
        return "float"
    if t == "string" or t.startswith("varchar") or t.startswith("char"):
        return "string"
    if t == "date":
        return "date"
    if t.startswith("timestamp"):
        return "timestamp"
    if t == "boolean":
        return "boolean"
    return "other"


def is_numeric_family(family: str) -> bool:
    return family in ("integer", "decimal", "float")


def decimal_scale(spark_type: str) -> int | None:
    m = re.match(r"decimal\((\d+),\s*(\d+)\)", spark_type.lower())
    return int(m.group(2)) if m else None


def _matches_any(name: str, patterns: list[str]) -> bool:
    return any(re.search(p, name) for p in patterns)


def is_id_name(column: str, cfg: SynthgenConfig) -> bool:
    return _matches_any(column.lower(), cfg.profiling.id_name_patterns)


def pii_name_matches(column: str, cfg: SynthgenConfig) -> list[str]:
    """Semantic types whose name patterns match the column name, in config order."""
    name = column.lower()
    return [sem for sem, pats in cfg.privacy.pii_name_patterns.items() if _matches_any(name, pats)]


def pii_tag_matches(tags: dict[str, str], cfg: SynthgenConfig) -> list[str]:
    hits = []
    for name, value in tags.items():
        candidates = [name.lower(), f"{name}={value}".lower()]
        if any(fnmatch.fnmatch(c, p.lower()) for c in candidates for p in cfg.privacy.pii_tag_patterns):
            hits.append(f"{name}={value}" if value else name)
    return hits


def config_marks_sensitive(full_column: str, cfg: SynthgenConfig) -> bool:
    return any(fnmatch.fnmatch(full_column.lower(), p.lower()) for p in cfg.privacy.sensitive_columns)


@dataclass
class Sensitivity:
    sensitive: bool
    reasons: list[str] = field(default_factory=list)
    semantic_hint: str | None = None


def static_sensitivity(table: str, column: str, tags: dict[str, str], cfg: SynthgenConfig) -> Sensitivity:
    """Sensitivity known before reading data: UC tags, name heuristics, config."""
    reasons: list[str] = []
    for hit in pii_tag_matches(tags, cfg):
        reasons.append(f"uc_tag:{hit}")
    name_hits = pii_name_matches(column, cfg)
    reasons.extend(f"name:{sem}" for sem in name_hits)
    if config_marks_sensitive(f"{table}.{column}", cfg):
        reasons.append("config")
    return Sensitivity(bool(reasons), reasons, name_hits[0] if name_hits else None)


def value_pii_reasons(regex_shares: dict[str, float], id_named: bool, cfg: SynthgenConfig) -> list[str]:
    """Reasons from value patterns. Phone is skipped for identifier columns (numeric-looking ids)."""
    out = []
    for sem in VALUE_PII_TYPES:
        if sem == "phone" and id_named:
            continue
        if regex_shares.get(sem, 0.0) >= cfg.privacy.value_pii_min_share:
            out.append(f"value_pattern:{sem}")
    return out


def guess_semantic_type(
    *,
    name_hint: str | None,
    regex_shares: dict[str, float],
    id_named: bool,
    mean_length: float | None,
    space_share: float,
    cfg: SynthgenConfig,
) -> str:
    if name_hint:
        return name_hint
    best = max(regex_shares.items(), key=lambda kv: kv[1], default=(None, 0.0))
    if best[0] and best[1] >= cfg.privacy.value_pii_min_share:
        return best[0]
    if id_named:
        return "identifier"
    if mean_length is not None and mean_length >= cfg.profiling.free_text_min_mean_length and space_share >= 0.5:
        return "free_text"
    if mean_length is not None:
        return "short_text"
    return "unknown"


def string_is_categorical(distinct: int, ratio: float | None, cfg: SynthgenConfig) -> bool:
    p = cfg.profiling
    if ratio is None:
        return True  # all null: nothing to export either way
    return (
        distinct <= p.max_categorical_distinct
        and ratio <= p.max_categorical_ratio
        and ratio <= p.high_cardinality_ratio
    )


# --- foreign key name matching -------------------------------------------------------------

_KEY_SUFFIX = re.compile(r"(_?(id|key|uuid|no|nr|code))$")


def singular(word: str) -> str:
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith(("ses", "xes", "ches", "shes")):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss") and len(word) > 3:
        return word[:-1]
    return word


def normalize_key_name(name: str) -> str:
    return _KEY_SUFFIX.sub("", name.lower()).strip("_")


def fk_name_score(child_column: str, parent_table: str, parent_column: str) -> float:
    """How strongly a child column name suggests a reference to parent_table.parent_column (0..1)."""
    child = child_column.lower()
    pcol = parent_column.lower()
    ptable = singular(parent_table.split(".")[-1].lower())
    stem = normalize_key_name(child)
    if child == pcol and pcol != "id":
        return 1.0
    if stem == ptable:
        return 0.95
    if stem.endswith("_" + ptable):  # e.g. referred_by_customer_id -> customers
        return 0.85
    if stem and stem == normalize_key_name(pcol):
        return 0.9
    if not stem:
        return 0.0
    return max(SequenceMatcher(None, stem, ptable).ratio(), SequenceMatcher(None, stem, normalize_key_name(pcol)).ratio())
''')

_load("synthgen.profiler.stats", r'''
"""Small helpers shared by profiler modules: k-suppression, rounding, column references, conversions."""

from __future__ import annotations

import math
from datetime import date, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

from pyspark.sql import Column
from pyspark.sql import functions as F


def col(name: str) -> Column:
    return F.col("`" + name.replace("`", "``") + "`")


def rnd(x: float | int | None, digits: int = 6) -> float | None:
    if x is None:
        return None
    x = float(x)
    if math.isnan(x) or math.isinf(x):
        return None
    return round(x, digits)


def safe_share(count: int | None, total: int | None, k: int) -> float | None:
    """count/total, or None when 0 < count < k (k-suppression). 0 stays 0."""
    if count is None or not total:
        return None
    if count == 0:
        return 0.0
    if count < k:
        return None
    return rnd(count / total)


def share_or_suppress(name: str, count: int | None, total: int | None, k: int, suppressed: list[str]) -> float | None:
    value = safe_share(count, total, k)
    if value is None and count:
        suppressed.append(name)
    return value


def qkey(q: float) -> str:
    return f"{q:g}"


def sum_if(cond: Column) -> Column:
    return F.sum(F.when(cond, 1).otherwise(0))


EPOCH_DATE = date(1970, 1, 1)
EPOCH_TS = datetime(1970, 1, 1, tzinfo=timezone.utc)


def days_to_iso(days: float | None) -> str | None:
    return None if days is None else (EPOCH_DATE + timedelta(days=int(days))).isoformat()


def micros_to_iso(micros: float | None, *, ntz: bool, session_tz: str) -> str | None:
    """UTC timestamps render as ...Z; TIMESTAMP_NTZ renders as local wall time without offset."""
    if micros is None:
        return None
    instant = EPOCH_TS + timedelta(microseconds=int(micros))
    if ntz:
        return instant.astimezone(ZoneInfo(session_tz)).replace(tzinfo=None, microsecond=0).isoformat()
    return instant.replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")
''')

_load("synthgen.profiler.metadata", r'''
"""Table metadata: names, types, comments, partitions, tags and declared keys.

Two providers:
- `SparkCatalogMetadata` uses only the Spark catalog API (works anywhere, no tags or constraints).
- `UnityCatalogMetadata` adds tags and declared PK/FK constraints from `<catalog>.information_schema`.

Both only read catalog metadata, never table rows.
"""

from __future__ import annotations

import fnmatch
import logging
import re
from dataclasses import dataclass, field

from pyspark.sql import SparkSession

from synthgen.config import SourceConfig, SynthgenConfig
from synthgen.profile_schema import DeclaredForeignKey
from synthgen.profiler._safe import collect_metadata

log = logging.getLogger(__name__)

_IDENT = re.compile(r"^[A-Za-z0-9_\-]+$")
_MAX_META_ROWS = 100_000


def quote(*parts: str) -> str:
    for p in parts:
        if not _IDENT.match(p):
            raise ValueError(f"unsupported identifier {p!r}")
    return ".".join(f"`{p}`" for p in parts)


def split_name(full_name: str) -> tuple[str, str, str]:
    parts = full_name.split(".")
    if len(parts) != 3:
        raise ValueError(f"expected catalog.schema.table, got {full_name!r}")
    return parts[0], parts[1], parts[2]


@dataclass
class ColumnMeta:
    name: str
    type: str
    nullable: bool
    comment: str | None
    is_partition: bool
    tags: dict[str, str] = field(default_factory=dict)


@dataclass
class TableMeta:
    full_name: str
    comment: str | None
    columns: list[ColumnMeta]
    tags: dict[str, str] = field(default_factory=dict)
    primary_key: list[str] | None = None
    foreign_keys: list[DeclaredForeignKey] = field(default_factory=list)

    @property
    def partition_columns(self) -> list[str]:
        return [c.name for c in self.columns if c.is_partition]


class SparkCatalogMetadata:
    name = "spark"

    def __init__(self, spark: SparkSession):
        self.spark = spark

    # --- table discovery ------------------------------------------------------------------

    def _schemas(self, catalog: str, pattern: str) -> list[str]:
        if not any(ch in pattern for ch in "*?["):
            return [pattern]
        rows = collect_metadata(self.spark.sql(f"SHOW SCHEMAS IN {quote(catalog)}"), max_rows=_MAX_META_ROWS, what="schemas")
        return sorted(r[0] for r in rows if fnmatch.fnmatch(r[0], pattern))

    def _tables(self, catalog: str, schema: str, include_views: bool) -> list[str]:
        out = []
        for t in self.spark.catalog.listTables(f"{catalog}.{schema}"):
            if t.isTemporary or (t.tableType == "VIEW" and not include_views):
                continue
            out.append(f"{catalog}.{schema}.{t.name}")
        return out

    def resolve_tables(self, source: SourceConfig) -> list[str]:
        found: list[str] = list(source.tables)
        for pattern in source.table_patterns:
            catalog, schema_pat, table_pat = split_name(pattern)
            for schema in self._schemas(catalog, schema_pat):
                for full in self._tables(catalog, schema, source.include_views):
                    if fnmatch.fnmatch(full, f"{catalog}.{schema}.{table_pat}"):
                        found.append(full)
        excluded = [t for t in found if any(fnmatch.fnmatch(t, p) for p in source.exclude_patterns)]
        return sorted(set(found) - set(excluded))

    # --- per table ------------------------------------------------------------------------

    def table_meta(self, full_name: str) -> TableMeta:
        table = self.spark.catalog.getTable(full_name)
        columns = [
            ColumnMeta(
                name=c.name,
                type=c.dataType,
                nullable=bool(c.nullable),
                comment=c.description or None,
                is_partition=bool(c.isPartition),
            )
            for c in self.spark.catalog.listColumns(full_name)
        ]
        meta = TableMeta(full_name=full_name, comment=table.description or None, columns=columns)
        self._enrich(meta)
        return meta

    def _enrich(self, meta: TableMeta) -> None:
        """Hook for providers that know more (tags, constraints)."""


class UnityCatalogMetadata(SparkCatalogMetadata):
    name = "unity_catalog"

    def _query(self, sql: str, args: dict[str, str], what: str):
        return collect_metadata(self.spark.sql(sql, args=args), max_rows=_MAX_META_ROWS, what=what)

    def _enrich(self, meta: TableMeta) -> None:
        cat, sch, tbl = split_name(meta.full_name)
        info = quote(cat, "information_schema")
        args = {"cat": cat, "sch": sch, "tbl": tbl}

        for r in self._query(
            f"SELECT tag_name, tag_value FROM {info}.table_tags "
            "WHERE catalog_name = :cat AND schema_name = :sch AND table_name = :tbl",
            args,
            "table tags",
        ):
            meta.tags[r["tag_name"]] = r["tag_value"] or ""

        by_col = {c.name: c for c in meta.columns}
        for r in self._query(
            f"SELECT column_name, tag_name, tag_value FROM {info}.column_tags "
            "WHERE catalog_name = :cat AND schema_name = :sch AND table_name = :tbl",
            args,
            "column tags",
        ):
            if r["column_name"] in by_col:
                by_col[r["column_name"]].tags[r["tag_name"]] = r["tag_value"] or ""

        key_rows = self._query(
            f"SELECT tc.constraint_name, tc.constraint_type, k.column_name, k.ordinal_position, "
            f"k.position_in_unique_constraint "
            f"FROM {info}.table_constraints tc JOIN {info}.key_column_usage k "
            "ON k.constraint_catalog = tc.constraint_catalog AND k.constraint_schema = tc.constraint_schema "
            "AND k.constraint_name = tc.constraint_name "
            "WHERE tc.table_catalog = :cat AND tc.table_schema = :sch AND tc.table_name = :tbl "
            "AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')",
            args,
            "key constraints",
        )
        pk = sorted((r for r in key_rows if r["constraint_type"] == "PRIMARY KEY"), key=lambda r: r["ordinal_position"])
        if pk:
            meta.primary_key = [r["column_name"] for r in pk]

        fk_names = sorted({r["constraint_name"] for r in key_rows if r["constraint_type"] == "FOREIGN KEY"})
        if not fk_names:
            return
        parent_rows = self._query(
            f"SELECT rc.constraint_name, p.table_catalog, p.table_schema, p.table_name, p.column_name, "
            f"p.ordinal_position "
            f"FROM {info}.referential_constraints rc JOIN {info}.key_column_usage p "
            "ON p.constraint_catalog = rc.unique_constraint_catalog "
            "AND p.constraint_schema = rc.unique_constraint_schema "
            "AND p.constraint_name = rc.unique_constraint_name "
            "WHERE rc.constraint_catalog = :cat AND rc.constraint_schema = :sch",
            {"cat": cat, "sch": sch},
            "referential constraints",
        )
        for name in fk_names:
            child = sorted(
                (r for r in key_rows if r["constraint_name"] == name and r["constraint_type"] == "FOREIGN KEY"),
                key=lambda r: r["ordinal_position"],
            )
            parents = {r["ordinal_position"]: r for r in parent_rows if r["constraint_name"] == name}
            if not parents:
                log.warning("FK %s on %s: parent key not visible in %s.information_schema", name, meta.full_name, cat)
                continue
            first = next(iter(parents.values()))
            parent_cols = [parents[r["position_in_unique_constraint"] or r["ordinal_position"]]["column_name"] for r in child]
            meta.foreign_keys.append(
                DeclaredForeignKey(
                    name=name,
                    columns=[r["column_name"] for r in child],
                    parent_table=f"{first['table_catalog']}.{first['table_schema']}.{first['table_name']}",
                    parent_columns=parent_cols,
                )
            )


def make_metadata_provider(spark: SparkSession, cfg: SynthgenConfig, catalogs: set[str] | None = None) -> SparkCatalogMetadata:
    if cfg.metadata_provider == "spark":
        return SparkCatalogMetadata(spark)
    if cfg.metadata_provider == "unity_catalog":
        return UnityCatalogMetadata(spark)
    catalogs = catalogs or {t.split(".")[0] for t in cfg.source.tables + cfg.source.table_patterns}
    has_info = all(spark.catalog.tableExists(f"{c}.information_schema.table_tags") for c in catalogs)
    provider = UnityCatalogMetadata(spark) if has_info else SparkCatalogMetadata(spark)
    log.info("metadata provider: %s", provider.name)
    return provider
''')

_load("synthgen.profiler.rules", r'''
"""Cheap within-table rules, detected from the data: date ordering and numeric identities.

Sensitive and identifier columns never take part in rules.
"""

from __future__ import annotations

from itertools import combinations

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

from synthgen.config import SynthgenConfig
from synthgen.profile_schema import DateOrderRule, NumericIdentityRule
from synthgen.profiler._safe import collect_single
from synthgen.profiler.stats import col, share_or_suppress, sum_if

# --- date ordering (evaluated inside the table's main aggregate) ----------------------------


def date_order_exprs(columns: list[tuple[int, str, str]]) -> list[Column]:
    """columns: (index, name, family) of non-sensitive date/timestamp columns."""
    exprs = []
    for (i, a, fa), (j, b, fb) in combinations(columns, 2):
        as_date = "date" in (fa, fb)
        ca = col(a).cast("date") if as_date else col(a)
        cb = col(b).cast("date") if as_date else col(b)
        both = ca.isNotNull() & cb.isNotNull()
        exprs += [
            sum_if(both).alias(f"do_{i}_{j}_n"),
            sum_if(both & (ca <= cb)).alias(f"do_{i}_{j}_le"),
            sum_if(both & (ca >= cb)).alias(f"do_{i}_{j}_ge"),
        ]
    return exprs


def date_order_rules(agg: dict, columns: list[tuple[int, str, str]], cfg: SynthgenConfig) -> list[DateOrderRule]:
    k = cfg.privacy.k
    rules = []
    for (i, a, _), (j, b, _) in combinations(columns, 2):
        n, le, ge = agg[f"do_{i}_{j}_n"], agg[f"do_{i}_{j}_le"], agg[f"do_{i}_{j}_ge"]
        if not n:
            continue
        for before, after, ok in ((a, b, le), (b, a, ge)):
            if ok / n >= cfg.rules.date_order_min_rate:
                suppressed: list[str] = []
                violation = share_or_suppress("match_rate", n - ok, n, k, suppressed)
                rate = None if violation is None else round(1 - violation, 6)
                rules.append(DateOrderRule(before=before, after=after, match_rate=rate, suppressed=suppressed))
                break
    return rules


# --- numeric identities (separate aggregate over a sample) ----------------------------------


def numeric_identity_rules(
    df: DataFrame, row_count: int, columns: list[str], cfg: SynthgenConfig, *, table: str
) -> list[NumericIdentityRule]:
    """Test a = b * c and a = b + c for every triple of the given numeric columns, on a sample."""
    rc = cfg.rules
    columns = columns[: rc.max_identity_columns]
    if len(columns) < 3 or row_count == 0:
        return []
    fraction = min(1.0, rc.identity_sample_rows / row_count)
    sample = df if fraction >= 1.0 else df.sample(fraction=fraction, seed=cfg.profiling.sample_seed)

    candidates: list[tuple[str, str, str]] = []
    exprs: list[Column] = []
    for a in columns:
        for b, c in combinations([x for x in columns if x != a], 2):
            xa, xb, xc = (col(x).cast("double") for x in (a, b, c))
            both = xa.isNotNull() & xb.isNotNull() & xc.isNotNull()
            tol = F.greatest(F.lit(rc.identity_abs_tolerance), F.abs(xa) * rc.identity_rel_tolerance)
            idx = len(candidates)
            exprs.append(sum_if(both).alias(f"n_{idx}"))
            for op, expr in (("product", xb * xc), ("sum", xb + xc)):
                exprs.append(sum_if(both & (F.abs(xa - expr) <= tol)).alias(f"{op}_{idx}"))
            candidates.append((a, b, c))
    agg = collect_single(sample.agg(*exprs), what=f"{table} numeric identities")

    rules = []
    for idx, (a, b, c) in enumerate(candidates):
        n = agg[f"n_{idx}"]
        if not n:
            continue
        for op in ("product", "sum"):
            ok = agg[f"{op}_{idx}"]
            if ok / n >= rc.identity_min_rate:
                suppressed: list[str] = []
                violation = share_or_suppress("match_rate", n - ok, n, cfg.privacy.k, suppressed)
                rate = None if violation is None else round(1 - violation, 6)
                rules.append(
                    NumericIdentityRule(
                        target=a, operation=op, operands=[b, c], match_rate=rate,
                        sample_fraction=round(fraction, 6), suppressed=suppressed,
                    )
                )
    return rules
''')

_load("synthgen.profiler.relationships", r'''
"""Primary key candidates and foreign keys with containment, orphan rate and children-per-parent stats."""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from datetime import datetime
from itertools import combinations

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

from synthgen import __version__
from synthgen.config import SynthgenConfig
from synthgen.profile_schema import ChildrenPerParent, DeclaredForeignKey, ForeignKey, PrimaryKey, Relationships
from synthgen.profiler._safe import collect_single
from synthgen.profiler.detect import fk_name_score, singular
from synthgen.profiler.stats import col, qkey, rnd, share_or_suppress, sum_if

log = logging.getLogger(__name__)


@dataclass
class KeyColumn:
    name: str
    family: str
    sensitive: bool
    id_named: bool


@dataclass
class KeyInfo:
    """What relationship detection needs to know about one profiled table."""

    table: str
    df: DataFrame  # full table, never sampled
    row_count: int
    columns: dict[str, KeyColumn]
    declared_pk: list[str] | None = None
    declared_fks: list[DeclaredForeignKey] = field(default_factory=list)
    unique_singles: list[str] = field(default_factory=list)
    unique_pairs: list[tuple[str, str]] = field(default_factory=list)


# --- primary keys -----------------------------------------------------------------------------


def find_unique_keys(
    df: DataFrame,
    row_count: int,
    singles: list[str],
    pair_pool: list[str],
    cfg: SynthgenConfig,
    *,
    table: str,
) -> tuple[list[str], list[tuple[str, str]]]:
    """Exact uniqueness checks on the full table. Pairs are tested only if no single column is unique."""
    if row_count == 0:
        return [], []
    found: list[str] = []
    if singles:
        exprs = []
        for i, name in enumerate(singles):
            exprs += [F.countDistinct(col(name)).alias(f"d{i}"), F.count(col(name)).alias(f"n{i}")]
        agg = collect_single(df.agg(*exprs), what=f"{table} key uniqueness")
        found = [n for i, n in enumerate(singles) if agg[f"d{i}"] == agg[f"n{i}"] == row_count]
    if found:
        return found, []
    pool = pair_pool[: cfg.relationships.max_pk_pair_columns]
    pairs = list(combinations(pool, 2))
    if not pairs:
        return [], []
    exprs = []
    for i, (a, b) in enumerate(pairs):
        exprs += [
            F.countDistinct(col(a), col(b)).alias(f"d{i}"),
            sum_if(col(a).isNotNull() & col(b).isNotNull()).alias(f"n{i}"),
        ]
    agg = collect_single(df.agg(*exprs), what=f"{table} key pair uniqueness")
    return [], [p for i, p in enumerate(pairs) if agg[f"d{i}"] == agg[f"n{i}"] == row_count]


def _single_pk_rank(table: str, column: KeyColumn) -> tuple[int, str]:
    name = column.name.lower()
    stem = singular(table.split(".")[-1].lower())
    if name in (f"{stem}_id", "id"):
        return (0, name)
    if column.id_named:
        return (1, name)
    return (2, name)


def choose_primary_key(info: KeyInfo, cfg: SynthgenConfig) -> PrimaryKey | None:
    forced = cfg.relationships.forced_pks.get(info.table)
    uniques = [[c] for c in sorted(info.unique_singles, key=lambda c: _single_pk_rank(info.table, info.columns[c]))]
    uniques += [list(p) for p in info.unique_pairs]
    if info.declared_pk:
        return PrimaryKey(table=info.table, columns=info.declared_pk, source="declared",
                          alternatives=[u for u in uniques if u != info.declared_pk])
    if forced:
        return PrimaryKey(table=info.table, columns=forced, source="config",
                          alternatives=[u for u in uniques if u != forced])
    if uniques:
        return PrimaryKey(table=info.table, columns=uniques[0], source="inferred", alternatives=uniques[1:])
    return None


# --- foreign keys -----------------------------------------------------------------------------


def fk_stats(child: DataFrame, child_col: str, parent: DataFrame, parent_col: str, cfg: SynthgenConfig, *, what: str) -> dict:
    """Aggregate-only statistics for a single-column FK. Returns raw counts plus quantiles."""
    child_counts = (
        child.select(col(child_col).alias("k")).where(F.col("k").isNotNull()).groupBy("k").agg(F.count(F.lit(1)).alias("n"))
    )
    parent_keys = parent.select(col(parent_col).alias("k")).where(F.col("k").isNotNull()).distinct()

    totals = collect_single(
        child_counts.agg(F.count(F.lit(1)).alias("values"), F.sum("n").alias("rows")), what=f"{what} child totals"
    )
    orphans = collect_single(
        child_counts.join(parent_keys, "k", "left_anti").agg(
            F.count(F.lit(1)).alias("values"), F.sum("n").alias("rows")
        ),
        what=f"{what} orphans",
    )
    qs = cfg.relationships.children_quantiles
    per_parent = (
        parent_keys.join(child_counts, "k", "left").select(F.coalesce(F.col("n"), F.lit(0)).alias("n"))
    )
    cpp = collect_single(
        per_parent.agg(
            F.count(F.lit(1)).alias("parents"),
            sum_if(F.col("n") == 0).alias("zero"),
            F.when(
                F.count(F.lit(1)) >= cfg.privacy.min_count_for_distribution,
                F.percentile_approx(F.col("n").cast("double"), qs, cfg.profiling.percentile_accuracy),
            ).alias("q"),
        ),
        what=f"{what} children per parent",
    )
    return {
        "child_values": totals["values"] or 0,
        "child_rows": totals["rows"] or 0,
        "orphan_values": orphans["values"] or 0,
        "orphan_rows": orphans["rows"] or 0,
        "parents": cpp["parents"] or 0,
        "zero_parents": cpp["zero"] or 0,
        "quantiles": {qkey(q): rnd(v) for q, v in zip(qs, cpp["q"])} if cpp["q"] else {},
    }


def _fk_from_stats(base: dict, stats: dict, cfg: SynthgenConfig) -> ForeignKey:
    k = cfg.privacy.k
    suppressed: list[str] = []
    orphan_rate = share_or_suppress("orphan_rate", stats["orphan_rows"], stats["child_rows"], k, suppressed)
    orphan_values = share_or_suppress("orphan_value_share", stats["orphan_values"], stats["child_values"], k, suppressed)
    zero_share = share_or_suppress(
        "children_per_parent.zero_children_share", stats["zero_parents"], stats["parents"], k, suppressed
    )
    return ForeignKey(
        **base,
        containment=None if orphan_rate is None else round(1 - orphan_rate, 6),
        orphan_rate=orphan_rate,
        orphan_value_share=orphan_values,
        children_per_parent=ChildrenPerParent(quantiles=stats["quantiles"], zero_children_share=zero_share),
        suppressed=suppressed,
    )


def _containment(stats: dict) -> float:
    """Row-based: share of non-null child rows whose value exists in the parent key."""
    return 1 - stats["orphan_rows"] / stats["child_rows"] if stats["child_rows"] else 0.0


def _split_column(full: str) -> tuple[str, str]:
    table, _, column = full.rpartition(".")
    return table, column


def infer_relationships(tables: dict[str, KeyInfo], cfg: SynthgenConfig, generated_at: datetime) -> Relationships:
    pks = {name: pk for name, info in tables.items() if (pk := choose_primary_key(info, cfg))}
    fks: list[ForeignKey] = []
    claimed: set[tuple[str, str]] = set()  # (child_table, child_column) already linked

    def explicit(child_table: str, child_cols: list[str], parent_table: str, parent_cols: list[str], source: str) -> None:
        base = dict(child_table=child_table, child_columns=child_cols, parent_table=parent_table,
                    parent_columns=parent_cols, source=source)
        claimed.update((child_table, c) for c in child_cols)
        if len(child_cols) != 1:
            fks.append(ForeignKey(**base, warnings=["composite foreign key: statistics not computed in pass 1"]))
            return
        if parent_table not in tables:
            fks.append(ForeignKey(**base, warnings=["parent table not profiled: statistics not computed"]))
            return
        stats = fk_stats(tables[child_table].df, child_cols[0], tables[parent_table].df, parent_cols[0], cfg,
                         what=f"{child_table}.{child_cols[0]}")
        fks.append(_fk_from_stats(base, stats, cfg))

    for name, info in tables.items():
        for d in info.declared_fks:
            explicit(name, d.columns, d.parent_table, d.parent_columns, "declared")
    for forced in cfg.relationships.forced_fks:
        (ct, cc), (pt, pc) = _split_column(forced.child), _split_column(forced.parent)
        if (ct, cc) not in claimed and ct in tables:
            explicit(ct, [cc], pt, [pc], "config")

    # Inference: single-column parent keys only.
    parents = [(t, pk.columns[0]) for t, pk in pks.items() if len(pk.columns) == 1]
    for child_table, info in tables.items():
        child_pk = pks.get(child_table)
        for column in info.columns.values():
            if column.sensitive or (child_table, column.name) in claimed:
                continue
            best: tuple[float, float, str, str, dict] | None = None
            for parent_table, parent_col in parents:
                if parent_table == child_table and child_pk and child_pk.columns == [column.name]:
                    continue  # a table's own single-column key
                pcol = tables[parent_table].columns[parent_col]
                if pcol.family != column.family:
                    continue
                score = fk_name_score(column.name, parent_table, parent_col)
                if score < cfg.relationships.name_similarity_threshold:
                    continue
                stats = fk_stats(info.df, column.name, tables[parent_table].df, parent_col, cfg,
                                 what=f"{child_table}.{column.name}")
                containment = _containment(stats)
                log.info("FK candidate %s.%s -> %s.%s score=%.2f", child_table, column.name, parent_table, parent_col, score)
                if containment >= cfg.relationships.fk_min_containment and (
                    best is None or (containment, score) > (best[0], best[1])
                ):
                    best = (containment, score, parent_table, parent_col, stats)
            if best:
                _, score, parent_table, parent_col, stats = best
                base = dict(child_table=child_table, child_columns=[column.name], parent_table=parent_table,
                            parent_columns=[parent_col], source="inferred", name_score=round(score, 3))
                fks.append(_fk_from_stats(base, stats, cfg))
                claimed.add((child_table, column.name))

    return Relationships(
        generated_at=generated_at,
        profiler_version=__version__,
        k_threshold=cfg.privacy.k,
        tables=sorted(tables),
        primary_keys=[pks[t] for t in sorted(pks)],
        foreign_keys=sorted(fks, key=lambda f: (f.child_table, f.child_columns)),
    )
''')

_load("synthgen.profiler.pass1", r'''
"""Pass 1: generic profiler. Knows nothing about business questions.

Per table, Spark runs:
1. one aggregate over the (optionally sampled) table for all column statistics and date-order rules;
2. one frequency pass (explode + group by) for category labels, format patterns and date/hour buckets,
   with k-suppression and top-N applied inside Spark so rare values never reach the driver;
3. small aggregates for exact key uniqueness and numeric identities.
Then relationships are inferred across tables and roles are finalised before anything is written.

Privacy rules applied here (and re-checked by the privacy guard in milestone 2):
- only aggregates are collected, through `_safe`;
- every count below k is suppressed (rate set to null, field listed in `suppressed`);
- sensitive, identifier and high-cardinality columns export no values;
- min/max are clipped quantiles, mean/std are computed on values capped at them;
  distributions need `min_count_for_distribution` non-null rows.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql import functions as F

from synthgen import __version__
from synthgen.config import SynthgenConfig
from synthgen.io import ProfileStore
from synthgen.profile_schema import (
    BooleanStats,
    CategoricalStats,
    ColumnProfile,
    DateStats,
    LengthStats,
    NumericStats,
    PatternShare,
    Relationships,
    TableProfile,
    TextStats,
    ValueShare,
)
from synthgen.profiler import detect
from synthgen.profiler._safe import collect_aggregate, collect_single
from synthgen.profiler.metadata import ColumnMeta, SparkCatalogMetadata, TableMeta, make_metadata_provider
from synthgen.profiler.relationships import KeyColumn, KeyInfo, find_unique_keys, infer_relationships
from synthgen.profiler.rules import date_order_exprs, date_order_rules, numeric_identity_rules
from synthgen.profiler.stats import (
    col,
    days_to_iso,
    micros_to_iso,
    qkey,
    rnd,
    share_or_suppress,
    sum_if,
)

log = logging.getLogger(__name__)

DOW_NAMES = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
DOW_ORDER = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


class ExportPolicyError(RuntimeError):
    """Raised when a profile would export something the hard constraints forbid."""


@dataclass
class _Col:
    meta: ColumnMeta
    idx: int
    family: str
    id_named: bool
    static: detect.Sensitivity
    sensitive: bool = False
    reasons: list[str] = field(default_factory=list)
    kind: str = "none"  # numeric | categorical | text | date | boolean | none
    role: str = "other"

    @property
    def name(self) -> str:
        return self.meta.name

    @property
    def stringish(self) -> bool:
        """Gets length (and, for strings, value-regex) analysis."""
        return self.family != "other" and (self.family == "string" or self.static.sensitive or self.id_named)

    @property
    def value_stats_allowed(self) -> bool:
        return not (self.static.sensitive or self.id_named)


@dataclass
class TableResult:
    profile: TableProfile
    keys: KeyInfo
    cols: dict[str, _Col]


@dataclass
class Pass1Result:
    profiles: list[TableProfile]
    relationships: Relationships
    written: list[Path]


# --- aggregate expressions --------------------------------------------------------------------


def _gate(c: Column, cfg: SynthgenConfig, agg: Column) -> Column:
    """Evaluate a distribution aggregate only when enough non-null rows exist (checked inside Spark)."""
    return F.when(F.count(c) >= cfg.privacy.min_count_for_distribution, agg)


def _quantile_list(cfg: SynthgenConfig) -> list[float]:
    return sorted(set(cfg.profiling.quantiles) | {cfg.privacy.clip_lower, cfg.privacy.clip_upper})


def _column_exprs(c: _Col, cfg: SynthgenConfig) -> list[Column]:
    p, acc = f"c{c.idx}_", cfg.profiling.percentile_accuracy
    x = col(c.name)
    exprs = [F.count(x).alias(p + "nn")]
    if c.family != "other":
        exprs.append(F.approx_count_distinct(x, cfg.profiling.approx_distinct_rsd).alias(p + "nd"))

    if c.stringish:
        s = x.cast("string")
        exprs += [
            _gate(x, cfg, F.percentile_approx(F.length(s), [cfg.privacy.clip_lower, cfg.privacy.clip_upper], acc)).alias(p + "len_q"),
            _gate(x, cfg, F.avg(F.length(s))).alias(p + "len_mean"),
        ]
        if c.family == "string":
            exprs += [sum_if(s.rlike(rx)).alias(f"{p}rx_{sem}") for sem, rx in detect.VALUE_REGEXES.items()]
            exprs.append(sum_if(s.rlike(r"\s")).alias(p + "space"))

    if not c.value_stats_allowed:
        return exprs

    if detect.is_numeric_family(c.family):
        v = x.cast("double")
        exprs += [
            _gate(x, cfg, F.percentile_approx(v, _quantile_list(cfg), acc)).alias(p + "q"),
            sum_if(v == 0).alias(p + "zero"),
            sum_if(v < 0).alias(p + "neg"),
        ]
        if c.family != "integer":
            exprs += [
                sum_if(v != F.floor(v)).alias(p + "nonint"),
                F.max(F.length(F.regexp_extract(x.cast("string"), r"\.(\d*?)0*$", 1))).alias(p + "dec"),
            ]
    elif c.family in ("date", "timestamp"):
        d = F.unix_date(x) if c.family == "date" else F.unix_micros(x.cast("timestamp"))
        exprs.append(_gate(x, cfg, F.percentile_approx(d, _quantile_list(cfg), acc)).alias(p + "q"))
        if c.family == "timestamp":
            ts = x.cast("timestamp")
            exprs.append(sum_if(ts != F.date_trunc("day", ts)).alias(p + "nonmidnight"))
    elif c.family == "boolean":
        exprs += [sum_if(x == F.lit(True)).alias(p + "t"), sum_if(x == F.lit(False)).alias(p + "f")]
    return exprs


def _clipped_moments(df: DataFrame, cols: list[_Col], agg: dict, cfg: SynthgenConfig, table: str) -> None:
    """Mean and std of numeric columns with values capped at the clip quantiles, so a few extreme
    values cannot inflate them. Needs the quantiles first, hence a second aggregate. Updates agg."""
    lo_i, hi_i = (_quantile_list(cfg).index(q) for q in (cfg.privacy.clip_lower, cfg.privacy.clip_upper))
    exprs = []
    for c in cols:
        q = agg.get(f"c{c.idx}_q")
        if c.kind != "numeric" or not q:
            continue
        v = F.least(F.greatest(col(c.name).cast("double"), F.lit(q[lo_i])), F.lit(q[hi_i]))
        exprs += [F.avg(v).alias(f"c{c.idx}_mean"), F.stddev_samp(v).alias(f"c{c.idx}_std")]
    if exprs:
        agg.update(collect_single(df.agg(*exprs), what=f"{table} clipped moments"))


def _pattern(s: Column, cfg: SynthgenConfig) -> Column:
    masked = F.regexp_replace(F.regexp_replace(s, r"\p{L}", "A"), r"\p{Nd}", "9")
    return F.when(F.length(s) <= cfg.profiling.pattern_max_length, masked)


def _frequency_items(cols: list[_Col], cfg: SynthgenConfig) -> tuple[list[Column], dict[str, int]]:
    items: list[Column] = []

    def item(c: _Col, kind: str, key: Column) -> None:
        items.append(F.struct(F.lit(c.idx).alias("c"), F.lit(kind).alias("k"), key.cast("string").alias("v")))

    for c in cols:
        x = col(c.name)
        if c.kind == "categorical":
            item(c, "v", x)
        elif c.kind == "text":
            item(c, "p", _pattern(x.cast("string"), cfg))
        elif c.kind == "date":
            item(c, "ym", F.date_format(x, "yyyy-MM"))
            item(c, "dow", F.dayofweek(x))
            if c.family == "timestamp":
                item(c, "h", F.lpad(F.hour(x).cast("string"), 2, "0"))
    limits = {"v": cfg.privacy.top_k_values, "p": cfg.profiling.pattern_top_n,
              "ym": cfg.profiling.max_year_months, "dow": 7, "h": 24}
    return items, limits


def _frequencies(df: DataFrame, cols: list[_Col], cfg: SynthgenConfig, table: str):
    """Returns (kept, buckets): kept[(idx, kind)] = [(key, count)] with count >= k, top-N by count;
    buckets[(idx, kind)] = number of distinct keys seen. Rare keys never leave Spark."""
    items, limits = _frequency_items(cols, cfg)
    if not items:
        return {}, {}
    k = cfg.privacy.k
    grouped = (
        df.select(F.explode(F.array(*items)).alias("e"))
        .select("e.c", "e.k", "e.v")
        .where(F.col("v").isNotNull())
        .groupBy("c", "k", "v")
        .agg(F.count(F.lit(1)).alias("n"))
    )
    limit = F.lit(0)
    for kind, n in limits.items():
        limit = F.when(F.col("k") == kind, F.lit(n)).otherwise(limit)
    rank = F.row_number().over(Window.partitionBy("c", "k").orderBy(F.col("n").desc(), F.col("v")))
    top = grouped.where(F.col("n") >= k).withColumn("r", rank).where(F.col("r") <= limit).select("c", "k", "v", "n")

    max_rows = sum(limits[r] for r in limits) * len(cols)
    kept: dict[tuple[int, str], list[tuple[str, int]]] = {}
    for r in collect_aggregate(top, max_rows=max_rows, what=f"{table} frequencies"):
        if r["n"] < k:  # defence in depth: Spark already filtered
            raise ExportPolicyError(f"{table}: bucket below k reached the driver")
        kept.setdefault((r["c"], r["k"]), []).append((r["v"], r["n"]))
    buckets = {
        (r["c"], r["k"]): r["b"]
        for r in collect_aggregate(
            grouped.groupBy("c", "k").agg(F.count(F.lit(1)).alias("b")), max_rows=len(items), what=f"{table} bucket counts"
        )
    }
    for v in kept.values():
        v.sort(key=lambda kv: (-kv[1], kv[0]))
    return kept, buckets


# --- per-column decisions and stats -----------------------------------------------------------


def _decide(c: _Col, agg: dict, cfg: SynthgenConfig) -> None:
    p = f"c{c.idx}_"
    nn = agg[p + "nn"]
    reasons = list(c.static.reasons)
    if c.family == "string" and nn:
        shares = {sem: agg[f"{p}rx_{sem}"] / nn for sem in detect.VALUE_REGEXES}
        reasons += [r for r in detect.value_pii_reasons(shares, c.id_named, cfg) if r not in reasons]
    c.reasons, c.sensitive = reasons, bool(reasons)

    if c.family == "other":
        c.kind, c.role = "none", "other"
    elif c.sensitive or c.id_named:
        c.kind = "text"
        c.role = {"date": "date", "timestamp": "date", "boolean": "boolean"}.get(
            c.family, "numeric" if detect.is_numeric_family(c.family) and not c.id_named else "text"
        )
    elif c.family == "boolean":
        c.kind = c.role = "boolean"
    elif detect.is_numeric_family(c.family):
        c.kind = c.role = "numeric"
    elif c.family in ("date", "timestamp"):
        c.kind = c.role = "date"
    else:
        nd = min(agg.get(p + "nd") or 0, nn)
        categorical = detect.string_is_categorical(nd, nd / nn if nn else None, cfg)
        c.kind = c.role = "categorical" if categorical else "text"


def _length(agg: dict, p: str) -> LengthStats:
    q = agg.get(p + "len_q")
    return LengthStats(
        min_clipped=rnd(q[0]) if q else None,
        mean=rnd(agg.get(p + "len_mean")),
        max_clipped=rnd(q[1]) if q else None,
    )


def _shares(pairs: list[tuple[str, int]], total: int) -> tuple[list[tuple[str, float]], float]:
    shares = [(key, rnd(n / total)) for key, n in pairs]
    kept = sum(n for _, n in pairs)
    return shares, max(0.0, rnd(1 - kept / total)) if total else 0.0


def _numeric_stats(c: _Col, agg: dict, nn: int, cfg: SynthgenConfig, suppressed: list[str]) -> NumericStats:
    p, k = f"c{c.idx}_", cfg.privacy.k
    qs, q = _quantile_list(cfg), agg[f"c{c.idx}_q"]
    quantiles = {qkey(a): rnd(v) for a, v in zip(qs, q)} if q else {}
    nonint = agg.get(p + "nonint", 0)
    is_integer = c.family == "integer" or nonint == 0
    cap = detect.decimal_scale(c.meta.type)
    cap = cfg.profiling.max_decimals if cap is None else min(cap, cfg.profiling.max_decimals)
    return NumericStats(
        min_clipped=quantiles.get(qkey(cfg.privacy.clip_lower)),
        max_clipped=quantiles.get(qkey(cfg.privacy.clip_upper)),
        mean=rnd(agg.get(p + "mean")),
        std=rnd(agg.get(p + "std")),
        quantiles={key: v for key, v in quantiles.items() if float(key) in cfg.profiling.quantiles},
        zero_share=share_or_suppress("numeric.zero_share", agg[p + "zero"], nn, k, suppressed),
        negative_share=share_or_suppress("numeric.negative_share", agg[p + "neg"], nn, k, suppressed),
        is_integer=is_integer,
        decimals=0 if is_integer else min(int(agg.get(p + "dec") or 0), cap),
    )


def _date_stats(c: _Col, agg: dict, nn: int, kept, cfg: SynthgenConfig, session_tz: str) -> DateStats:
    p = f"c{c.idx}_"
    ntz = c.meta.type.lower() == "timestamp_ntz"
    if c.family == "date":
        conv = days_to_iso
    else:
        def conv(v):
            return micros_to_iso(v, ntz=ntz, session_tz=session_tz)
    qs, q = _quantile_list(cfg), agg[p + "q"]
    quantiles = {qkey(a): conv(v) for a, v in zip(qs, q)} if q else {}
    ym, ym_other = _shares(sorted(kept.get((c.idx, "ym"), [])), nn)
    dow = {DOW_NAMES[int(key)]: s for key, s in _shares(kept.get((c.idx, "dow"), []), nn)[0]}
    stats = DateStats(
        min_clipped=quantiles.get(qkey(cfg.privacy.clip_lower)),
        max_clipped=quantiles.get(qkey(cfg.privacy.clip_upper)),
        quantiles={key: v for key, v in quantiles.items() if float(key) in cfg.profiling.quantiles},
        year_month_shares=dict(ym),
        other_year_month_share=ym_other,
        day_of_week_shares={d: dow[d] for d in DOW_ORDER if d in dow},
    )
    if c.family == "timestamp":
        stats.hour_shares = dict(sorted(_shares(kept.get((c.idx, "h"), []), nn)[0]))
        stats.always_midnight = agg[p + "nonmidnight"] == 0
        stats.timezone = None if ntz else session_tz
    return stats


def _column_profile(c: _Col, agg: dict, n: int, kept, buckets, cfg: SynthgenConfig, session_tz: str) -> ColumnProfile:
    p, k = f"c{c.idx}_", cfg.privacy.k
    nn = agg[p + "nn"]
    nd = min(agg.get(p + "nd") or 0, nn)
    suppressed: list[str] = []
    prof = ColumnProfile(
        name=c.name,
        position=c.idx,
        type=c.meta.type,
        nullable=c.meta.nullable,
        comment=c.meta.comment,
        tags=c.meta.tags,
        is_partition=c.meta.is_partition,
        role=c.role,
        sensitive=c.sensitive,
        sensitive_reasons=c.reasons,
        null_rate=share_or_suppress("null_rate", n - nn, n, k, suppressed),
        distinct_approx=nd,
        distinct_ratio=rnd(nd / nn) if nn else None,
    )
    if c.kind == "numeric":
        prof.numeric = _numeric_stats(c, agg, nn, cfg, suppressed)
    elif c.kind == "date":
        prof.date = _date_stats(c, agg, nn, kept, cfg, session_tz)
    elif c.kind == "boolean":
        prof.boolean = BooleanStats(
            true_share=share_or_suppress("boolean.true_share", agg[p + "t"], n, k, suppressed),
            false_share=share_or_suppress("boolean.false_share", agg[p + "f"], n, k, suppressed),
        )
    elif c.kind == "categorical":
        pairs = kept.get((c.idx, "v"), [])
        values, other = _shares(pairs, nn)
        prof.categorical = CategoricalStats(
            top_values=[ValueShare(value=v, share=s) for v, s in values],
            other_share=other,
            labels_not_exported=max(0, buckets.get((c.idx, "v"), 0) - len(pairs)),
            length=_length(agg, p),
        )
    elif c.kind == "text":
        patterns, other = _shares(kept.get((c.idx, "p"), []), nn)
        shares = {sem: agg[f"{p}rx_{sem}"] / nn for sem in detect.VALUE_REGEXES} if c.family == "string" and nn else {}
        value_hint = next((r.split(":")[1] for r in c.reasons if r.startswith("value_pattern:")), None)
        length = _length(agg, p)
        prof.text = TextStats(
            length=length,
            patterns=[PatternShare(pattern=v, share=s) for v, s in patterns],
            other_pattern_share=other if nn else 0.0,
            semantic_type=detect.guess_semantic_type(
                name_hint=c.static.semantic_hint or value_hint,
                regex_shares=shares,
                id_named=c.id_named,
                mean_length=length.mean,
                space_share=agg.get(p + "space", 0) / nn if nn else 0.0,
                cfg=cfg,
            ),
        )
    prof.suppressed = suppressed
    return prof


# --- export policy ------------------------------------------------------------------------------

_VALUE_FIELDS = ("numeric", "categorical", "date", "boolean")


def enforce_export_policy(profile: TableProfile, id_named: set[str]) -> None:
    """Last check before writing: columns that must not export values carry no value statistics."""
    for c in profile.columns:
        no_values = c.sensitive or c.name in id_named or c.role in ("key", "foreign_key", "text", "other")
        if no_values and any(getattr(c, f) is not None for f in _VALUE_FIELDS):
            raise ExportPolicyError(f"{profile.source_table}.{c.name}: value statistics on a no-values column")
        if c.role == "other" and c.text is not None:
            raise ExportPolicyError(f"{profile.source_table}.{c.name}: unsupported type exported text stats")


# --- table ------------------------------------------------------------------------------------


def profile_table(spark: SparkSession, meta: TableMeta, cfg: SynthgenConfig, generated_at: datetime) -> TableResult:
    name = meta.full_name
    log.info("profiling %s", name)
    full = spark.table(name)
    row_count = full.count()
    fraction = cfg.sample_fraction_for(name)
    df = full if fraction >= 1.0 else full.sample(fraction=fraction, seed=cfg.profiling.sample_seed)
    session_tz = spark.conf.get("spark.sql.session.timeZone")

    cols = [
        _Col(
            meta=m,
            idx=i,
            family=detect.type_family(m.type),
            id_named=detect.is_id_name(m.name, cfg),
            static=detect.static_sensitivity(name, m.name, m.tags, cfg),
        )
        for i, m in enumerate(meta.columns)
    ]
    date_cols = [(c.idx, c.name, c.family) for c in cols if c.family in ("date", "timestamp") and c.value_stats_allowed]

    exprs = [F.count(F.lit(1)).alias("_n")]
    for c in cols:
        exprs += _column_exprs(c, cfg)
    exprs += date_order_exprs(date_cols)
    agg = collect_single(df.agg(*exprs), what=f"{name} column statistics")
    n = agg["_n"]

    for c in cols:
        _decide(c, agg, cfg)
    _clipped_moments(df, cols, agg, cfg, name)
    kept, buckets = _frequencies(df, cols, cfg, name)

    warnings: list[str] = []
    if fraction < 1.0:
        warnings.append(f"column statistics computed on a {fraction:g} sample; row_count is exact")
    for c in cols:
        if c.family == "other":
            warnings.append(f"column {c.name} has unsupported type {c.meta.type}: only type and null rate exported")

    rules = date_order_rules(agg, date_cols, cfg)
    numeric_cols = [c.name for c in cols if c.kind == "numeric"]
    rules += numeric_identity_rules(full, row_count, numeric_cols, cfg, table=name)

    profile = TableProfile(
        generated_at=generated_at,
        profiler_version=__version__,
        source_table=name,
        row_count=row_count,
        sample_fraction=fraction,
        k_threshold=cfg.privacy.k,
        comment=meta.comment,
        tags=meta.tags,
        primary_key=meta.primary_key,
        foreign_keys=meta.foreign_keys,
        partition_columns=meta.partition_columns,
        columns=[_column_profile(c, agg, n, kept, buckets, cfg, session_tz) for c in cols],
        rules=rules,
        warnings=warnings,
    )

    # Key candidates: exact uniqueness is checked on the full table, prefiltered by approx distinct.
    def key_ok(c: _Col) -> bool:
        nn = agg[f"c{c.idx}_nn"]
        return not c.sensitive and c.family in ("integer", "string") and n > 0 and nn == n

    singles = [
        c.name for c in cols
        if key_ok(c) and (agg.get(f"c{c.idx}_nd") or 0) >= cfg.relationships.pk_prefilter_ratio * n
    ]
    pair_pool = [c.name for c in cols if key_ok(c) and c.id_named]
    unique_singles, unique_pairs = find_unique_keys(full, row_count, singles, pair_pool, cfg, table=name)

    keys = KeyInfo(
        table=name,
        df=full,
        row_count=row_count,
        columns={c.name: KeyColumn(c.name, c.family, c.sensitive, c.id_named) for c in cols},
        declared_pk=meta.primary_key,
        declared_fks=list(meta.foreign_keys),
        unique_singles=unique_singles,
        unique_pairs=unique_pairs,
    )
    return TableResult(profile=profile, keys=keys, cols={c.name: c for c in cols})


def _apply_key_roles(results: dict[str, TableResult], rel: Relationships) -> None:
    """Key and FK columns are identifiers: set their role and drop any value statistics."""

    def mark(table: str, column: str, role: str) -> None:
        res = results.get(table)
        if not res:
            return
        for c in res.profile.columns:
            if c.name != column:
                continue
            if any(getattr(c, f) is not None for f in _VALUE_FIELDS):
                for f in _VALUE_FIELDS:
                    setattr(c, f, None)
                c.suppressed = [s for s in c.suppressed if s == "null_rate"]
                res.profile.warnings.append(f"value statistics of key column {column} not exported")
            c.role = role

    for pk in rel.primary_keys:
        for column in pk.columns:
            mark(pk.table, column, "key")
    for fk in rel.foreign_keys:
        for column in fk.child_columns:
            mark(fk.child_table, column, "foreign_key")


def run_pass1(
    spark: SparkSession,
    cfg: SynthgenConfig,
    *,
    metadata: SparkCatalogMetadata | None = None,
    write: bool = True,
) -> Pass1Result:
    provider = metadata or make_metadata_provider(spark, cfg)
    tables = provider.resolve_tables(cfg.source)
    if not tables:
        raise ValueError("no source tables matched the configuration")
    generated_at = datetime.now(timezone.utc).replace(microsecond=0)

    results = {t: profile_table(spark, provider.table_meta(t), cfg, generated_at) for t in tables}
    rel = infer_relationships({t: r.keys for t, r in results.items()}, cfg, generated_at)
    _apply_key_roles(results, rel)

    profiles = [results[t].profile for t in tables]
    for t in tables:
        enforce_export_policy(results[t].profile, {c.name for c in results[t].cols.values() if c.id_named})

    written: list[Path] = []
    if write:
        store = ProfileStore.review(cfg.output.root)
        written = [store.write_table_profile(p) for p in profiles] + [store.write_relationships(rel)]
    return Pass1Result(profiles=profiles, relationships=rel, written=written)
''')

print("synthgen engine loaded:", sys.modules["synthgen"].__version__)

In [ ]:
# ============================== 4. RUN: profile the tables =======================================
import logging

from synthgen.notebook_helpers import config_from_notebook
from synthgen.profiler.pass1 import run_pass1

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
for noisy in ("py4j", "pyspark"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

cfg = config_from_notebook(TABLES, OUTPUT_FOLDER, EXTRA_SENSITIVE_COLUMNS, ADVANCED_SETTINGS)
result = run_pass1(spark, cfg)

# Summary only: names, counts and file paths. This notebook never displays table data.
for profile in result.profiles:
    sensitive = [c.name for c in profile.columns if c.sensitive]
    print(f"{profile.source_table}: {profile.row_count} rows, {len(profile.columns)} columns, "
          f"values hidden for: {', '.join(sensitive) or 'none'}")
print(f"\n{len(result.relationships.foreign_keys)} links between tables found")
print("\nFiles to review:")
for path in result.written:
    print("  ", path)

## 5. Review the files before sending

Open the `review/` folder inside your output folder: one `.json` file per table plus `relationships.json`.

- **Sensitive columns** (`"sensitive": true`) show only length and a format such as `"AAAA.AAAA@AAAA.AAA"`
  (letters become `A`, digits `9`). No values.
- **Categorical columns** list their most common values with shares, e.g. `"value": "France", "share": 0.74`.
  Values used by fewer than 10 rows are never listed. **Main thing to check:** are you comfortable sending
  these category names (countries, product categories, statuses, channels...)?
- **Comments**: table and column descriptions from Unity Catalog are copied as they are. Check they contain
  no example personal data or confidential text.
- **Numbers and dates** show percentiles and shares by month, weekday and hour. Extreme values are cut off.

If something must not leave, **do not edit the JSON**: add the column to `EXTRA_SENSITIVE_COLUMNS`
(or remove the table from `TABLES`) in cell 2 and run the notebook again.

In [ ]:
# ============================== 6. PACKAGE: run after your review ================================
# Zips exactly the files in review/ into synthgen_profiles.zip next to it. Send only that zip.
# Download it from Catalog Explorer (volume) or by right-clicking the file (workspace folder).
import shutil
from pathlib import Path

archive = shutil.make_archive(str(Path(cfg.output.root) / "synthgen_profiles"), "zip", Path(cfg.output.root) / "review")
print(archive)